In [1]:
import os
import pycolmap
import numpy as np
from tqdm.notebook import tqdm
from mylib.geometry import compute_relative_camera_motion

# pairs_calibrated.txt is used for pose estimation benchmarks
# views.txt is used only for repeatability benchmarks

In [8]:
data_path = 'benchmarks_2D/terrasky3d/data'
scenes = sorted(os.listdir(data_path))
# scenes if scene is a directory]  # filter only valid
scenes = [scene for scene in scenes if os.path.isdir(os.path.join(data_path, scene))]
scenes 

['graz_castle',
 'graz_church',
 'graz_clocktower',
 'graz_main_square',
 'graz_townhall',
 'graz_university',
 'udine_devils_bridge',
 'udine_fagagna_church',
 'udine_villalta_castle']

In [3]:
def build_k(camera):
    """ Build intrinsic matrix from pycolmap camera object """
    model = camera.model.name
    params = camera.params
    
    if model == 'SIMPLE_RADIAL':
        f, cx, cy, k = params
        K = np.array([[f, 0, cx],
                      [0, f, cy],
                      [0, 0, 1]])
    elif model == 'PINHOLE':
        fx, fy, cx, cy = params
        K = np.array([[fx, 0, cx],
                      [0, fy, cy],
                      [0, 0, 1]])
    elif model == 'SIMPLE_PINHOLE':
        f, cx, cy = params
        K = np.array([[f, 0, cx],
                      [0, f, cy],
                      [0, 0, 1]])
    else:
        raise NotImplementedError(f"Camera model {model} not implemented")
    
    return K

In [4]:
files = []
for scene_name in tqdm(scenes):
    rec_path = f"{data_path}/{scene_name}/colmap/sparse/0"
    if not os.path.exists(rec_path):
        print(f"Reconstruction path {rec_path} does not exist. Skipping scene {scene_name}.")
        continue
    rec = pycolmap.Reconstruction(rec_path)
    with open(f"{data_path}/{scene_name}/colmap/viewgraph_30.txt", "r") as f:
        lines = f.read().splitlines()
    pairs = [line.split() for line in lines]
    pairs = np.array(pairs)[:, :2]

    for pair in pairs:
        img1, img2 = pair[0].item(), pair[1].item()
        try:
            K1 = build_k(rec.find_image_with_name(img1).camera)
            K2 = build_k(rec.find_image_with_name(img2).camera)

            R1 = rec.find_image_with_name(img1).cam_from_world.rotation.matrix()
            t1 = rec.find_image_with_name(img1).cam_from_world.translation
            R2 = rec.find_image_with_name(img2).cam_from_world.rotation.matrix()
            t2 = rec.find_image_with_name(img2).cam_from_world.translation

            R, t = compute_relative_camera_motion(R1, t1, R2, t2)
            R, t = R.numpy(), t.numpy()

            # Store line
            files.append(f"{scene_name}/frames/{img1} {scene_name}/frames/{img2} " +
                         " ".join(map(str, K1.flatten())) + " " +
                         " ".join(map(str, K2.flatten())) + " " +
                         " ".join(map(str, R.flatten())) + " " +
                         " ".join(map(str, t.flatten())))

    
        except Exception as e:
            print(f"Error processing pair ({img1}, {img2}): {e}")
            continue



  0%|          | 0/10 [00:00<?, ?it/s]

Error processing pair (77/IMG_4175_frame_000001.jpg, 77/IMG_4175_frame_000002.jpg): 'NoneType' object has no attribute 'camera'
Error processing pair (77/IMG_4175_frame_000001.jpg, 77/IMG_4175_frame_000003.jpg): 'NoneType' object has no attribute 'camera'
Error processing pair (77/IMG_4175_frame_000001.jpg, 77/IMG_4175_frame_000004.jpg): 'NoneType' object has no attribute 'camera'
Error processing pair (77/IMG_4175_frame_000001.jpg, 77/IMG_4175_frame_000005.jpg): 'NoneType' object has no attribute 'camera'
Error processing pair (77/IMG_4175_frame_000001.jpg, 77/IMG_4175_frame_000006.jpg): 'NoneType' object has no attribute 'camera'
Error processing pair (77/IMG_4175_frame_000001.jpg, 77/IMG_4175_frame_000007.jpg): 'NoneType' object has no attribute 'camera'
Error processing pair (77/IMG_4175_frame_000001.jpg, 77/IMG_4175_frame_000008.jpg): 'NoneType' object has no attribute 'camera'
Error processing pair (77/IMG_4175_frame_000002.jpg, 77/IMG_4175_frame_000003.jpg): 'NoneType' object ha

In [7]:
with open(f"{data_path}/pairs_calibrated.txt", "w") as f:
    f.write("# paths, intrisics, relative pose\n# scene/image_1_path scene/image_2_path (fx1 0 cx1 0 fy1 cy1 0 0 1) (fx2 0 cx1 0 fy2 cy2 0 0 1) (R00 R01 R02 R10 R11 R12 R20 R21 R22) (tx ty tx)")
    f.write("\n".join(files))

In [6]:
.

SyntaxError: invalid syntax (1933637684.py, line 1)

## covert megadepth view or ait2ground to md1500 format

In [ ]:
# views.txt format
# image_path R(3x3).flatten() t(3).flatten() model W H fx fy cx cy

# pairs_calibrated.txt format
# image1_path image2_path K1(3x3).flatten() K2(3x3).flatten() R_rel(3x3).flatten() t_rel(3).flatten()

In [ ]:
import numpy as np
from PIL import Image

path = Path("benchmarks/megadepth_air2ground/data")

index = np.load(path / "indices.npz", allow_pickle=True)

In [ ]:
index['pair_info']

In [ ]:
path = Path("benchmarks/megadepth_air2ground/data")

out = {}
for pair in index['pair_info']:
    img1_name = pair['pair_names'][0]
    if img1_name not in out:
        pose1 = pair['pose'][0]
        R1 = pose1[:3, :3]
        t1 = pose1[:3, 3]
        K1 = pair['intrinsic'][0]
        fx1, fy1, cx1, cy1 = K1[0, 0], K1[1, 1], K1[0, 2], K1[1, 2]
        model = "PINHOLE"
        w, h = Image.open(path / 'images' / img1_name).size

        s1 = f"{img1_name} {' '.join(map(str, R1.flatten()))} {' '.join(map(str, t1.flatten()))} {model} {w} {h} {fx1} {fy1} {cx1} {cy1}"
        out[img1_name] = s1

    img2_name = pair['pair_names'][1]
    if img2_name not in out:
        pose2 = pair['pose'][1]
        R2 = pose2[:3, :3]
        t2 = pose2[:3, 3]
        K2 = pair['intrinsic'][1]
        fx2, fy2, cx2, cy2 = K2[0, 0], K2[1, 1], K2[0, 2], K2[1, 2]
        model = "PINHOLE"
        img2_name = pair['pair_names'][1]
        w, h = Image.open(path / 'images' / img2_name).size

        s2 = f"{img2_name} {' '.join(map(str, R2.flatten()))} {' '.join(map(str, t2.flatten()))} {model} {w} {h} {fx2} {fy2} {cx2} {cy2}"
        out[img2_name] = s2

out = sorted(out.values())
with open(path / "views.txt", "w") as f:
    f.write("\n".join(out) + "\n")

In [ ]:
out = []
for pair in index['pair_info']:
    img1_name = pair['pair_names'][0]
    pose1 = pair['pose'][0]
    R1 = pose1[:3, :3]
    t1 = pose1[:3, 3]
    K1 = pair['intrinsic'][0]

    img2_name = pair['pair_names'][1]
    pose2 = pair['pose'][1]
    R2 = pose2[:3, :3]
    t2 = pose2[:3, 3]
    K2 = pair['intrinsic'][1]

    out.append(f"{img1_name} {img2_name} {' '.join(map(str, K1.flatten()))} {' '.join(map(str, K2.flatten()))} {' '.join(map(str, (R2 @ R1.T).flatten()))} {' '.join(map(str, (t2 - R2 @ R1.T @ t1).flatten()))}")

out = sorted(out)

In [ ]:
out = sorted(out)
with open(path / "pairs_calibrated.txt", "w") as f:
    f.write("\n".join(out) + "\n")